# Azure OpenAI Data Augmentation

Reads `Data/Global_outlok.json`, creates 4 paraphrased Q&A pairs for each original pair, validates the model output with Pydantic, and saves the final conversation-format dataset.

In [ ]:
# %pip install -U openai pydantic python-dotenv tqdm

Create a `.env` file with these values:

```text
AZURE_OPENAI_ENDPOINT=https://your-resource.openai.azure.com/
AZURE_OPENAI_API_KEY=your-key
AZURE_OPENAI_API_VERSION=2024-10-21
AZURE_OPENAI_DEPLOYMENT=your-chat-deployment-name
```

In [1]:
import json
import os
import shutil
from pathlib import Path

from dotenv import load_dotenv
from openai import AzureOpenAI
from pydantic import BaseModel, Field
from tqdm.auto import tqdm

load_dotenv()

INPUT_PATH = Path("Data/Global_outlok.json")
OUTPUT_PATH = Path("Data/Global_outlok_augmented.json")
CHECKPOINT_PATH = Path("Data/Global_outlok_augmented_checkpoint.json")

N_PARAPHRASES = 4
INCLUDE_ORIGINALS = True
APPEND_TO_SOURCE = True

DEPLOYMENT = os.getenv("AZURE_OPENAI_DEPLOYMENT") or os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT")
client = AzureOpenAI(
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    api_version=os.getenv("AZURE_OPENAI_API_VERSION", "2024-10-21"),
)

if not DEPLOYMENT:
    raise ValueError("Set AZURE_OPENAI_DEPLOYMENT in .env")

/home/dudukunta.reddy/.local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
class QAPair(BaseModel):
    question: str = Field(..., min_length=3)
    answer: str = Field(..., min_length=3)


class ParaphraseBatch(BaseModel):
    pairs: list[QAPair] = Field(..., min_length=N_PARAPHRASES, max_length=N_PARAPHRASES)


def as_conversation(pair: QAPair) -> list[dict[str, str]]:
    return [
        {"role": "user", "content": pair.question.strip()},
        {"role": "assistant", "content": pair.answer.strip()},
    ]

In [3]:
data = json.loads(INPUT_PATH.read_text(encoding="utf-8"))
conversations = data["conversations"]

for i, conv in enumerate(conversations):
    assert len(conv) == 2, f"Bad conversation length at index {i}"
    assert conv[0]["role"] == "user" and conv[1]["role"] == "assistant", f"Bad roles at index {i}"

print(f"Loaded {len(conversations):,} Q&A pairs")
print(f"Will generate {len(conversations) * N_PARAPHRASES:,} paraphrased pairs")

Loaded 1,430 Q&A pairs
Will generate 5,720 paraphrased pairs


In [4]:
def paraphrase(question: str, answer: str) -> list[list[dict[str, str]]]:
    prompt = f"""
Create exactly {N_PARAPHRASES} paraphrased Q&A pairs from the original pair.

Rules:
- Preserve the same meaning and all facts.
- Keep every number, date, percentage, currency amount, country name, and organization name unchanged.
- Do not add new facts.
- Make the wording varied enough for supervised fine-tuning.

Original question: {question}
Original answer: {answer}
""".strip()

    completion = client.beta.chat.completions.parse(
        model=DEPLOYMENT,
        messages=[
            {"role": "system", "content": "You generate factual training data and return only structured output."},
            {"role": "user", "content": prompt},
        ],
        response_format=ParaphraseBatch,
        temperature=0.7,
    )
    batch = completion.choices[0].message.parsed
    return [as_conversation(pair) for pair in batch.pairs]

In [5]:
generated = {}
if CHECKPOINT_PATH.exists():
    generated = json.loads(CHECKPOINT_PATH.read_text(encoding="utf-8"))

for i, conv in enumerate(tqdm(conversations, desc="Augmenting")):
    key = str(i)
    if key in generated:
        continue
    generated[key] = paraphrase(conv[0]["content"], conv[1]["content"])
    CHECKPOINT_PATH.write_text(json.dumps(generated, indent=2, ensure_ascii=False), encoding="utf-8")

augmented = []
for i in range(len(conversations)):
    augmented.extend(generated[str(i)])

final_conversations = conversations + augmented if INCLUDE_ORIGINALS else augmented
print(f"Generated: {len(augmented):,}")
print(f"Final total: {len(final_conversations):,}")

Augmenting:   0%|          | 0/1430 [00:00<?, ?it/s]

Augmenting: 100%|██████████| 1430/1430 [56:30<00:00,  2.37s/it]  

Generated: 5,720
Final total: 7,150


In [6]:
if APPEND_TO_SOURCE:
    backup = INPUT_PATH.with_suffix(INPUT_PATH.suffix + ".bak")
    shutil.copy2(INPUT_PATH, backup)
    INPUT_PATH.write_text(json.dumps({"conversations": final_conversations}, indent=4, ensure_ascii=False), encoding="utf-8")
    print(f"Backed up original to {backup} and appended into {INPUT_PATH}")
else:
    OUTPUT_PATH.write_text(json.dumps({"conversations": final_conversations}, indent=4, ensure_ascii=False), encoding="utf-8")
    print(f"Wrote {OUTPUT_PATH}")

Backed up original to Global_outlok.json.bak and appended into Global_outlok.json
